# Estrategia de ofertas FDS - Pricing Justo

Notebook delgado: la logica de negocio (elegibilidad, score, cascada de
mecanica, guardrail economico, modelo economico, post-mortem) vive en
`backend/pricing/` como funciones reutilizables - este notebook solo
conecta a Snowflake, llama esas funciones y reporta resultados.

- `backend/pricing/catalog.py` - FULL_MASTER_CATALOG, VW_PRICING_DASHBOARD
- `backend/pricing/elasticity.py` - ATHENEA + fallback en cascada
- `backend/pricing/strategy.py` - toda la estrategia de ofertas
- `backend/pricing/postmortem.py` - vistas/tabla de medicion recurrente en SANDBOX

Ver el plan completo en `/home/alejomd17/.claude/plans/vamos-a-cambiar-de-glowing-plum.md`.

In [ ]:
import sys
sys.path.insert(0, "..")  # para que "backend" se resuelva desde src/

import pandas as pd
import matplotlib.pyplot as plt

from backend.pricing import strategy, postmortem
from backend.pricing.snowflake_conn import get_connection

## Parametros

In [ ]:
RUTA_OPORTUNIDAD = "../data/inputs/oportunidad.xlsx"
RUTA_DESCUENTOS  = "../data/inputs/Descuentos comerciales.xlsx"

# Fin de semana objetivo (ajustar cada vez que se corra para un nuevo FDS)
# CORREGIDO: la campana real fue 11-13 jul, no 10-12 jul (se habia subido
# con la fecha equivocada - ver la celda de limpieza justo antes de
# "Subir manualmente el plan real").
WEEKEND_INICIO = pd.Timestamp("2026-07-11")   # sabado
WEEKEND_FIN    = pd.Timestamp("2026-07-13")   # lunes

# Nombre de archivo unico por corrida (finde + fecha de corrida) - nunca
# sobreescribe una corrida anterior, ni de otro finde ni de este mismo.
RUTA_SALIDA = strategy.generar_ruta_salida(WEEKEND_INICIO, WEEKEND_FIN, carpeta="../data/output")
print(f"Se va a guardar en: {RUTA_SALIDA}")

## Conexion a Snowflake

SSO interactivo (`authenticator="externalbrowser"`) - al ejecutar la celda
se abre el navegador para el login de Google.

In [2]:
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print(cur.fetchone())

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C02t5osnf&SAMLRequest=lZJdb9owFIb%2FSuRdJ44TxjqLgPgYGqwbEYQxcWdiQ606durjNO2%2Fn8OH1F200i4iRfb7%2BjnnvGcweqlU8CwsSKMzRKIYBUKXhkt9ytC2mId3KADHNGfKaJGhVwFoNBwAq1RNx4170Gvx1AhwgX9IA%2B0uMtRYTQ0DCVSzSgB1Jd2Mf97TJIopAxDWeRy6WjhIz3pwrqYYt20btWlk7AkncRzj%2BCv2qk7yCb1B1B8zamucKY26WV58T%2B8gCI57HcIrPCG%2FGidSX0bwEeVwEQH9XhR5mK82BQrGt%2B6mRkNTCbsR9lmWYru%2BvxQAvoLldlOswv18QtKUJBFo0x4VexSlqerG%2BScj%2F4ePgmNlTtIPajHLUP0ouePfzOaw27nlcnGYrv5MEr61T6u7eb6XpeO7H4RU%2BzXv87QsUfD7FmvSxboAaMRCd2E6fxQn%2FTD%2BEpK0SFLaIzTtRYT09yiY%2BTClZu7svFXMytI02kF0MuakxLk%2Bg7skEix5PfKf5Nk0TtxnA%2FqILhtCz0w7%2FK%2B%2BB%2Fit9bppv%2FzwF7PcKFm%2BBnNjK%2Bbez4ZE5HwieXg8S6momFRjzq0A8BkpZdqpFcz5hXa2EQgPL9R%2FV3r4Fw%3D%3D&RelayState=ver%3A3-hint%3A7286493704326390-ETMsDgAAAZ9d24AkABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEIkGDL96WkYhu6dR0Vve5awAAACgJQbz13mu4EBCR1Z3M58VeLBN2sg5bSdefV

gio: https://accounts.google.com/o/saml2/idp?idpid=C02t5osnf&SAMLRequest=lZJdb9owFIb%2FSuRdJ44TxjqLgPgYGqwbEYQxcWdiQ606durjNO2%2Fn8OH1F200i4iRfb7%2BjnnvGcweqlU8CwsSKMzRKIYBUKXhkt9ytC2mId3KADHNGfKaJGhVwFoNBwAq1RNx4170Gvx1AhwgX9IA%2B0uMtRYTQ0DCVSzSgB1Jd2Mf97TJIopAxDWeRy6WjhIz3pwrqYYt20btWlk7AkncRzj%2BCv2qk7yCb1B1B8zamucKY26WV58T%2B8gCI57HcIrPCG%2FGidSX0bwEeVwEQH9XhR5mK82BQrGt%2B6mRkNTCbsR9lmWYru%2BvxQAvoLldlOswv18QtKUJBFo0x4VexSlqerG%2BScj%2F4ePgmNlTtIPajHLUP0ouePfzOaw27nlcnGYrv5MEr61T6u7eb6XpeO7H4RU%2BzXv87QsUfD7FmvSxboAaMRCd2E6fxQn%2FTD%2BEpK0SFLaIzTtRYT09yiY%2BTClZu7svFXMytI02kF0MuakxLk%2Bg7skEix5PfKf5Nk0TtxnA%2FqILhtCz0w7%2FK%2B%2BB%2Fit9bppv%2FzwF7PcKFm%2BBnNjK%2Bbez4ZE5HwieXg8S6momFRjzq0A8BkpZdqpFcz5hXa2EQgPL9R%2FV3r4Fw%3D%3D&RelayState=ver%3A3-hint%3A7286493704326390-ETMsDgAAAZ9d24AkABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEIkGDL96WkYhu6dR0Vve5awAAACgJQbz13mu4EBCR1Z3M58VeLBN2sg5bSdefV6WGEW96D8lNCpQAv8PIZeX0nSw24V%2FJhq1tV69s6uUynWIG%2F3OgEDuEJJMz0noRtQnbkZlFDKwX07hFifSe0YU1ep4z8m0ZH3uY

('jorge.moscoso@justo.mx', 'DATA_SCIENCE', 'GENERAL_USERS', 'MX_JUSTO_PROD')


### Limpieza: borrar el plan de la ventana incorrecta (10-12 jul)

La campana real fue **11-13 jul**, no 10-12 jul - se habia subido con la
fecha equivocada. Antes de volver a subir (con el Excel corregido,
`estrategia_fds_jul11_jul13_20260709.xlsx`), hay que borrar las filas de
`WKND_PROMO_PLAN` de la ventana incorrecta - si no, quedarian dos "campanas"
distintas (10-12 y 11-13) en la tabla, y `WKND_PLAN_VS_ACTUAL_V` mostraria
la primera como una campana WKND que en realidad nunca existio.

Esta celda solo se corre UNA VEZ, para corregir el error - no es parte del
flujo normal de subir un plan.

In [ ]:
VENTANA_INCORRECTA = (pd.Timestamp("2026-07-10").date(), pd.Timestamp("2026-07-12").date())
cur.execute(
    "DELETE FROM MX_JUSTO_PROD.SANDBOX.WKND_PROMO_PLAN WHERE CAMPAIGN_START = %s AND CAMPAIGN_END = %s",
    VENTANA_INCORRECTA,
)
print(f"Filas borradas de la ventana incorrecta (10-12 jul): {cur.rowcount}")

In [ ]:
RUTA_REAL = "../data/output/estrategia_fds_jul11_jul13_20260709.xlsx"

general = pd.read_excel(RUTA_REAL, sheet_name="GENRAL")
general = general[["STORE_ID", "SKU", "ESTRATEGIA", "PRECIO_OFERTA", "MARGEN_OFERTA_%", "GMV_PROY_DIA", "UTIL_PROY_DIA"]].copy()

def cargar_hoja_flat(sheet_name):
    df = pd.read_excel(RUTA_REAL, sheet_name=sheet_name)
    df[["SKU", "Nombre"]] = df["SKU + Nombre"].str.split(" - ", n=1, expand=True)
    df["SKU"] = pd.to_numeric(df["SKU"], errors="coerce")
    df = df.dropna(subset=["SKU"]).copy()
    df["SKU"] = df["SKU"].astype(int)
    df["ESTRATEGIA"] = df["DESCUENTO"].apply(lambda d: f"SPON: -{d*100:.0f}%")
    df["PRECIO_OFERTA"] = df["PRECIO FINAL C/PROMO"]
    df["MARGEN_OFERTA_%"] = df["MARGEN REAL C/PROMO"] * 100
    df["GMV_PROY_DIA"] = pd.NA
    df["UTIL_PROY_DIA"] = pd.NA
    return df[["STORE_ID", "SKU", "ESTRATEGIA", "PRECIO_OFERTA", "MARGEN_OFERTA_%", "GMV_PROY_DIA", "UTIL_PROY_DIA"]]

cervezas = cargar_hoja_flat("Cervezas")
limpieza = cargar_hoja_flat("Limpieza")

estrategia_real_df = pd.concat([general, cervezas, limpieza], ignore_index=True)
print(f"Total filas del plan real: {len(estrategia_real_df)}")

n_subidas = postmortem.subir_plan_y_crear_vistas(cur, estrategia_real_df, WEEKEND_INICIO, WEEKEND_FIN)
print(f"Plan REAL subido a SANDBOX.WKND_PROMO_PLAN: {n_subidas} filas (ventana 11-13 jul)")


## Construir la estrategia

Une todos los pasos (exclusion comercial, catalogo real, elegibilidad,
Tema Mundial, descuento maximo, elasticidad real + fallback, score,
cascada de mecanica, guardrail economico, confianza + backtesting, dia de
ejecucion, escenarios) y exporta el Excel final de una sola vez. El detalle
de cada paso esta documentado como docstring en `backend/pricing/strategy.py`.

In [ ]:
if postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    df = strategy.construir_estrategia(
        cur=cur,
        ruta_oportunidad=RUTA_OPORTUNIDAD,
        ruta_descuentos=RUTA_DESCUENTOS,
        ruta_salida=RUTA_SALIDA,
        weekend_inicio=WEEKEND_INICIO,
        weekend_fin=WEEKEND_FIN,
    )
else:
    df = None
    print("WEEKEND_FIN ya paso - no se construye una estrategia nueva para una ventana ya ejecutada.")
    print("Ver mas abajo la seccion de post-mortem para medir/subir lo que realmente se ejecuto.")

## Resultados finales

In [ ]:
if df is None:
    print("No se construyo una estrategia nueva (WEEKEND_FIN ya paso) - nada que reportar aqui.")
else:
    ofer = df[df["MECANICA"] != "Sin oferta"]

    print("=== ESTRATEGIA ===")
    print(df["MECANICA"].value_counts().to_string())
    print(f"\nOfertas: {len(ofer)} de {len(df)} ({len(ofer)/len(df)*100:.1f}%) | "
          f"descuento exhibido promedio: {ofer['DESC_EFECTIVO'].mean()*100:.1f}% | "
          f"margen minimo en promocion: {ofer['MARGEN_OFERTA'].min():.2f}%")
    print(f"SKUs que requieren aprobacion Comercial (no incluidos, d_max real > 30%): "
          f"{df['REQUIERE_APROBACION'].sum()}")

    print("\n=== PROYECCION ESCENARIO BASE ===")
    print(f"Unidades: {ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f} "
          f"({(ofer['Q1_DIA'].sum()/ofer['Q0_DIA'].sum()-1)*100:+.1f}%)")
    print(f"GMV:      ${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f} "
          f"({(ofer['GMV_PROY_DIA'].sum()/ofer['GMV_BASE_DIA'].sum()-1)*100:+.1f}%)")
    print(f"Utilidad: ${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f} "
          f"({(ofer['UTIL_PROY_DIA'].sum()/ofer['UTIL_BASE_DIA'].sum()-1)*100:+.1f}%)")

    print("\n=== POR DIA ===")
    print(ofer.groupby("DIA_EJECUCION").agg(Ofertas=("SKU", "count"),
          GMV_inc=("GMV_INC_DIA", "sum"), Util_inc=("UTIL_INC_DIA", "sum")).round(0).to_string())

    print("\n=== CONFIANZA DE LA PROYECCION (ofertas finales) ===")
    print(ofer["CONFIANZA_PROYECCION"].value_counts())

    mun = ofer[ofer["TEMA_MUNDIAL"]]
    print(f"\n=== MUNDIAL ===\nOfertas: {len(mun)} | GMV incremental: ${mun['GMV_INC_DIA'].sum():,.0f} | "
          f"Utilidad incremental: ${mun['UTIL_INC_DIA'].sum():,.0f}")

## Subir el plan a Snowflake

Dos momentos distintos, con reglas distintas:

1. **Construir la estrategia para el proximo finde** (el que no ha pasado
   todavia): la celda de abajo sube el plan automaticamente, sobreescribiendo
   cualquier intento anterior para esa misma ventana - solo importa la
   ultima version, porque nada se ha ejecutado aun.
2. **Medir una promo que ya paso**: la celda de arriba NO sube nada
   automatico (para no pisar el plan real con una reconstruccion de hoy).
   Hay que subir manualmente el Excel que de verdad se ejecuto - ver la
   celda "Subir manualmente" mas abajo.

Las vistas (`WKND_PROMO_RESULTS_V`, `WKND_PLAN_VS_ACTUAL_V`, `WKND_POSTMORTEM_PROMO_V`)
se crean/reemplazan cada vez que se sube un plan - son recurrentes, no hay
que tocarlas aparte.

In [ ]:
# 1. Auto-subida: solo si WEEKEND_FIN todavia no paso (plan de un finde futuro)
if postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    estrategia_df = df[df["MECANICA"] != "Sin oferta"].rename(columns={"MARGEN_OFERTA": "MARGEN_OFERTA_%"})
    n_subidas = postmortem.subir_plan_y_crear_vistas(cur, estrategia_df, WEEKEND_INICIO, WEEKEND_FIN)
    print(f"Plan subido a SANDBOX.WKND_PROMO_PLAN: {n_subidas} filas "
          f"(finde aun no ejecutado, se sobreescribe en cada corrida)")
else:
    print("WEEKEND_FIN ya paso - no se subio nada automatico para no pisar el plan real ejecutado.")
    print('Usa la celda "Subir manualmente el plan real de una promo ya pasada" de abajo.')

### Subir manualmente el plan real de una promo ya pasada

Si `WEEKEND_FIN` ya paso, esta celda busca en disco (por fecha de finde,
con `strategy.buscar_salida_historica`) el Excel que realmente se ejecuto y
lo sube a `WKND_PROMO_PLAN` - es la version que "gano" esa semana, no una
reconstruccion de hoy. Si `WEEKEND_FIN` no ha pasado, no hace nada (ya se
subio arriba, automatico).

In [ ]:
if not postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    try:
        ruta_historica = strategy.buscar_salida_historica(WEEKEND_INICIO, WEEKEND_FIN, carpeta="../data/output")
    except FileNotFoundError as e:
        print(e)
        print("No se sube nada - no existe un Excel para esta ventana.")
    else:
        print(f"Plan leido de: {ruta_historica}")
        estrategia_df_historica = pd.read_excel(ruta_historica, sheet_name="Estrategia FDS")
        n_subidas = postmortem.subir_plan_y_crear_vistas(cur, estrategia_df_historica, WEEKEND_INICIO, WEEKEND_FIN)
        print(f"Plan subido a SANDBOX.WKND_PROMO_PLAN: {n_subidas} filas")
else:
    print("WEEKEND_FIN todavia no paso - el plan ya se subio automaticamente arriba, no hace falta esta celda.")

### Medir una promo ya pasada

Cambia `PROMO_INICIO`/`PROMO_FIN` a la ventana que quieras medir (ej. el
FDS 3-5 jul que ya corrio) y correlo - responde que mecanica gano, separado
por si fue nuestra propuesta u otra promo que coincidio en la ventana.

In [ ]:
PROMO_INICIO = pd.Timestamp("2026-07-11")
PROMO_FIN    = pd.Timestamp("2026-07-13")

# Mismo filtro por defecto que el dashboard: solo lo nuestro (WKND) y lo
# que si se ejecuto (con_mecanica). Pasa origen=None, adopcion=None aqui
# si quieres ver TODO (incluye "Otra fuente" y SKUs sin mecanica real).
resumen_mecanica = postmortem.performance_por_mecanica(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
resumen_mecanica

In [ ]:
total_planeado = postmortem.contar_plan(cur, PROMO_INICIO, PROMO_FIN)

# resumen_adopcion NO filtra por origen/adopcion - necesita ver TODOS los
# origenes (incluye "Sin promo") para que el denominador de cada bucket sea
# correcto. Alimenta los KPIs de Planeado/Adopcion/tarjetas por origen del
# reporte HTML, igual que /summary del dashboard.
resumen_adopcion_df = postmortem.resumen_adopcion(cur, PROMO_INICIO, PROMO_FIN)
resumen_adopcion_df

,ORIGEN_CAMPANA,SKU_TIENDAS,SKU_TIENDAS_CON_PROMO_REAL
0,Sin promo,4371,0
1,Otra fuente,1269,1269
2,WKND,85,75


### Validar la tasa de redencion real (FACT_FULFILLMENT_LINE)

`MX_JUSTO_PROD.DM_CORE.FACT_FULFILLMENT_LINE` trae detalle a nivel de linea
de pedido con `IS_DISCOUNT`/`IS_BULK_APPLIED` - permite comparar la tasa de
redencion **real** contra los supuestos declarados en
`strategy.REDENCION` (35%/40%/55%). Si la diferencia es grande, vale la pena
recalibrar esos supuestos igual que se hizo con `MULT_PROMO` en el
backtesting de arriba.

In [ ]:
redencion_real = postmortem.validar_redencion_real(cur, PROMO_INICIO, PROMO_FIN)
redencion_real

,BULK_STRATEGY,BULK_RULE_BUY,BULK_RULE_PAY,UNIDADES_TOTALES,UNIDADES_CON_DESCUENTO,TIER,REDENCION_REAL,REDENCION_SUPUESTA,DIFERENCIA
4,NaN,NaN,NaN,5929368,155712,NaN,0.0263,NaN,NaN
5,SPON,1.0,1.0,1066331,1066331,sin_umbral,1.0000,1.00,0.00
7,BNSDP,1.0,1.0,5453,5453,sin_umbral,1.0000,1.00,0.00
8,BNSP,4.0,4.0,616,616,sin_umbral,1.0000,1.00,0.00
0,BNSP,2.0,2.0,468,468,sin_umbral,1.0000,1.00,0.00
11,BNSP,1.0,1.0,429,429,sin_umbral,1.0000,1.00,0.00
10,BNSP,3.0,3.0,300,300,sin_umbral,1.0000,1.00,0.00
3,BNGM,2.0,1.0,86,86,umbral_2,1.0000,0.55,0.45
6,BNGM,3.0,2.0,78,78,pack_3,1.0000,0.40,0.60
1,SPON,2.0,2.0,14,14,sin_umbral,1.0000,1.00,0.00


### SKUs mas vendidos de la campana

Mismo tratamiento que `performance_por_mecanica`: excluye "Sin promo" y
colapsa fragmentos por dia/mecanica a una fila por SKU+tienda.

In [ ]:
top_skus_df = postmortem.top_skus(cur, PROMO_INICIO, PROMO_FIN, n=20, origen="WKND", adopcion="con_mecanica")
top_skus_df

,SKU,STORE_ID,DEPARTAMENTO,CATEGORIA,ORIGEN_CAMPANA,UNIDADES_TOTALES,GMV_TOTAL,MECANICA_EJECUTADA
2,10185.00000,14,Congelados,Frutas y Verduras Congeladas,WKND,77.0,2328.06,BNSDP
4,10458.00000,14,Despensa,"Sopas, Pastas y Purés",WKND,57.0,590.07,BNSDP
57,22383.00000,14,Limpieza y Hogar,Limpieza General,WKND,34.0,321.12,SPON
7,11306.00000,14,Despensa,"Sopas, Pastas y Purés",WKND,32.0,355.89,5x4
62,25308.00000,14,Bebidas,Jugos y Bebidas de Fruta,WKND,25.0,1107.24,BNSDP
32,14639.00000,14,Bebidas,Agua,WKND,24.0,248.19,SPON
25,14058.00000,14,Bebidas,Agua,WKND,23.0,1612.03,6x5
34,15579.00000,14,Limpieza y Hogar,Limpieza General,WKND,23.0,458.85,SPON
29,14342.00000,14,Limpieza y Hogar,Limpieza General,WKND,22.0,684.36,SPON
1,10185.00000,9,Congelados,Frutas y Verduras Congeladas,WKND,18.0,566.01,BNSDP


### Performance por plataforma y por tipo de cliente

`MARKETPLACE` (justo/express/uber/rappi/didi) y `SEGMENTO_USUARIO`
(clasificacion oficial de Justo - Recurrente/Reactivado/New, via
`MASTER_ORDER.USER_STATUS_ORDER_DELIVERED`) ya vienen en
`WKND_POSTMORTEM_PROMO_V` desde el 2026-07-13. Responde "¿en que
plataforma vendio mas la campana?" y "¿funciono mejor en recurrentes o en
reactivados?".

In [ ]:
marketplace_df = postmortem.resumen_por_marketplace(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
marketplace_df

In [ ]:
segmento_usuario_df = postmortem.resumen_por_segmento_usuario(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
segmento_usuario_df

### Gráfica: GMV por plataforma (pastel)

Mismos datos de `marketplace_df` de arriba, en pastel para ver de un
vistazo el peso relativo de cada canal.

In [ ]:
# Paleta "Weekly Pricing" (skill justo-brand-guide) - orden fijo por
# plataforma, no ciclado. No corrido por el validador de contraste de la
# skill dataviz (esto es exploratorio, no un artefacto que se vaya a
# publicar) - si esta grafica se vuelve permanente, correr esa validacion.
COLORES_PLATAFORMA = {
    "justo": "#158158", "express": "#058dc7", "uber": "#ed561b",
    "rappi": "#24cbe5", "didi": "#64e572",
}

mp = marketplace_df.dropna(subset=["GMV_TOTAL"]).sort_values("GMV_TOTAL", ascending=False)
colores = [COLORES_PLATAFORMA.get(p, "#888888") for p in mp["MARKETPLACE"]]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(mp["GMV_TOTAL"], labels=mp["MARKETPLACE"], autopct="%1.1f%%", colors=colores, startangle=90, counterclock=False)
ax.set_title("GMV por plataforma")
plt.show()

### Gráficas: GMV, ticket por unidad, unidades y margen por categoría y por departamento

Reusa `resumen_mecanica` (ya calculado arriba) - no vuelve a consultar
Snowflake. Ayuda a probar la hipótesis de "el ticket alto en Uber/Rappi es
por mezcla de categorías" (si las categorías con ticket alto coinciden con
lo que más se vende en esas plataformas, la explicación es mezcla, no
necesariamente markup).

In [ ]:
categoria_df = postmortem.resumen_por_categoria(resumen_mecanica)
departamento_df = postmortem.resumen_por_departamento(resumen_mecanica)

METRICAS = [
    ("GMV_TOTAL", "GMV ($)"),
    ("TICKET_POR_UNIDAD", "Ticket por unidad ($)"),
    ("UNIDADES_TOTALES", "Unidades"),
    ("MARGEN_PROMEDIO", "Margen (%)"),
]

for nombre_col, df in [("CATEGORIA", categoria_df), ("DEPARTAMENTO", departamento_df)]:
    df_ordenado = df.sort_values("GMV_TOTAL", ascending=True)
    fig, axes = plt.subplots(1, 4, figsize=(22, max(4, 0.35 * len(df_ordenado))))
    for ax, (campo, titulo) in zip(axes, METRICAS):
        ax.barh(df_ordenado[nombre_col], df_ordenado[campo], color="#158158")
        ax.set_title(titulo)
    fig.suptitle(f"Performance por {nombre_col.lower()}")
    plt.tight_layout()
    plt.show()

### Cruces categoria/departamento x plataforma/tipo de cliente

Para las barras agrupadas del dashboard/reporte (una serie por plataforma o
tipo de cliente, un grupo por categoria/departamento) - responde si el
patron de una categoria se explica por mezcla de plataforma/segmento o es
parejo en todas.

In [ ]:
categoria_plataforma_df = postmortem.resumen_por_categoria_y_plataforma(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
departamento_plataforma_df = postmortem.resumen_por_departamento_y_plataforma(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
categoria_segmento_df = postmortem.resumen_por_categoria_y_segmento(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
departamento_segmento_df = postmortem.resumen_por_departamento_y_segmento(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
categoria_plataforma_df

### Cruce plataforma x tipo de cliente

Prueba directa de la hipotesis "Sin dato en tipo de cliente es en realidad
marketplace externo": si casi todo el "Sin dato" cae en uber/rappi/didi (no
en justo/express), la hipotesis se confirma.

In [ ]:
plataforma_segmento_df = postmortem.resumen_por_plataforma_y_segmento(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
plataforma_segmento_df.pivot(index="MARKETPLACE", columns="SEGMENTO_USUARIO", values="GMV_TOTAL")

### Usuarios distintos y ticket promedio por usuario, por segmento

El dato que faltaba para comparar segmentos sin el sesgo de tamaño de
grupo: cuanta GENTE distinta hay en cada uno, no solo cuanto GMV/unidades
sumaron.

In [ ]:
usuarios_segmento_df = postmortem.resumen_usuarios_por_segmento(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
usuarios_segmento_df

### GMV, margen y volumen por mecánica x plataforma x tipo de cliente

Tabla granular a proposito (no grafica) - filtra/ordena para preguntas
puntuales tipo "¿el 5x4 en Uber le fue mejor a Recurrentes o a Nuevos?".

In [ ]:
descuento_plataforma_segmento_df = postmortem.resumen_descuento_plataforma_segmento(
    cur, PROMO_INICIO, PROMO_FIN, origen="WKND", adopcion="con_mecanica"
)
descuento_plataforma_segmento_df

### Reporte HTML del post-mortem (siempre se genera)

Junta todo lo de arriba (`resumen_mecanica`, `top_skus_df`, `resumen_adopcion_df`,
`total_planeado`, `marketplace_df`, `segmento_usuario_df`, `categoria_df`,
`departamento_df`, `plataforma_segmento_df`, `usuarios_segmento_df`,
`descuento_plataforma_segmento_df`) en un solo reporte HTML autonomo (mismo
estilo, mismos KPIs/graficas/tablas que el dashboard), guardado en
`data/output/`. Correr esta celda es el ultimo paso estandar de cualquier
post-mortem, no un entregable manual aparte.

In [ ]:
ruta_reporte = postmortem.generar_reporte_html(
    resumen_mecanica, top_skus_df, PROMO_INICIO, PROMO_FIN,
    resumen_df=resumen_adopcion_df, total_planeado=total_planeado,
    marketplace_df=marketplace_df, segmento_usuario_df=segmento_usuario_df,
    categoria_df=categoria_df, departamento_df=departamento_df,
    categoria_plataforma_df=categoria_plataforma_df, departamento_plataforma_df=departamento_plataforma_df,
    categoria_segmento_df=categoria_segmento_df, departamento_segmento_df=departamento_segmento_df,
    plataforma_segmento_df=plataforma_segmento_df, usuarios_segmento_df=usuarios_segmento_df,
    descuento_plataforma_segmento_df=descuento_plataforma_segmento_df,
    ruta_salida=f"../data/output/postmortem_{PROMO_INICIO.strftime('%b%d').lower()}_{PROMO_FIN.strftime('%b%d').lower()}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.html",
)
print(f"Reporte generado en: {ruta_reporte}")